# 01 - Demand EDA (T2.1.1)

**Data:** Corporacion Favorita (Kaggle), chain-wide weekly unit sales (sum over 54 stores), Monday weeks 2013-01-07 .. 2017-03-27 (ADR-005).
**SKU selection rule (reproduced by `src/shelflife/data/favorita.py`):** perishable = 1; families DAIRY, DELI, PREPARED FOODS; item has non-zero sales in >= 95% of full weeks up to 2017-03-27; sells in each of the final 13 weeks (not being phased out); top 4 per family by total units. **Demand unconstraining:** a week below 20% of the trailing 8-week median is a supply/data gap and is replaced by that median (past-only; raw kept in `units_raw`). Each plant SKU (configs/plant.yaml) borrows the weekly pattern of one anonymous Favorita item from the same family (assumption A-01: retail sales as a proxy for manufacturer demand). Mapping: `configs/skus.yaml`.
Rebuild: `python -m shelflife.data.favorita` (needs the raw Kaggle files).

In [1]:
import numpy as np, pandas as pd, yaml, matplotlib.pyplot as plt, matplotlib.dates as mdates
from pathlib import Path
from shelflife import viz
from shelflife.data import favorita as F
viz.apply_style()
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
panel = pd.read_parquet(root / 'data/processed/demand_weekly.parquet')
plant = {s['id']: s for s in yaml.safe_load((root / 'configs/plant.yaml').read_text(encoding='utf-8'))['skus']}
print(panel.shape, panel.week_start.min().date(), panel.week_start.max().date(), panel.sku_id.nunique(), 'SKUs')

(2652, 8) 2013-01-07 2017-03-27 12 SKUs


## Per-SKU weekly demand (small multiples; each panel has its own y-scale)

In [2]:
fig, axes = plt.subplots(4, 3, figsize=(12, 9.5), sharex=True)
for ax, (sku, g) in zip(axes.T.flat, panel.groupby('sku_id')):
    fam = g.family.iloc[0]
    ax.plot(g.week_start, g.units / 1000, color=viz.FAMILY_COLOR[fam], lw=1.2)
    imp = g[g.gap_imputed]
    if len(imp):
        ax.scatter(imp.week_start, imp.units / 1000, s=22, facecolor=viz.SURFACE, edgecolor=viz.INK_2, lw=1, zorder=3)
    ax.set_title(f"{sku} {plant[sku]['name']}  |  {fam.title()} item {g.item_nbr.iloc[0]}", fontsize=8.5, loc='left')
    ax.set_ylim(bottom=0)
    ax.xaxis.set_major_locator(mdates.YearLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
for ax in axes[:, 0]:
    ax.set_ylabel('thousand units / week')
fig.suptitle('Weekly demand per SKU (chain-wide Favorita sales; columns = Dairy, Deli, Prepared foods)', x=0.01, ha='left', fontsize=11)
viz.source_note(fig, 'Source: Corporacion Favorita (Kaggle). Plant SKU names are hypothetical; demand patterns are real. '
                     'Open circles = supply/data-gap weeks replaced by the trailing 8-week median (demand unconstraining).')
fig.tight_layout(rect=(0, 0.02, 1, 0.97))
fig.savefig(root / 'results/figures/demand_small_multiples.png', dpi=150)
plt.show()

C:\Users\Gideon\AppData\Local\Temp\ipykernel_18332\1620620687.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Seasonality: mean seasonal index by week of year, per family
Index = units / that SKU's mean in the same calendar year, averaged over SKUs and years, 3-week centred rolling mean. 1.0 = an average week.

In [3]:
p = panel.copy()
p['year'] = p.week_start.dt.isocalendar().year
p['woy'] = p.week_start.dt.isocalendar().week.clip(upper=52)
p['idx'] = p.units / p.groupby(['sku_id', 'year']).units.transform('mean')
full_years = p.groupby('year').week_start.nunique().loc[lambda s: s >= 50].index
si = (p[p.year.isin(full_years)].groupby(['family', 'woy']).idx.mean().unstack(0)
        .rolling(3, center=True, min_periods=1).mean())
fig, ax = plt.subplots(figsize=(10, 4.2))
for fam in si.columns:
    ax.plot(si.index, si[fam], color=viz.FAMILY_COLOR[fam], label=fam.title())
    ax.annotate(fam.title(), (si.index[-1], si[fam].iloc[-1]), xytext=(6, 0), textcoords='offset points',
                color=viz.INK_2, fontsize=8, va='center')
ax.axhline(1, color=viz.AXIS, lw=0.8)
ax.set_xlim(1, 57); ax.set_xlabel('ISO week of year'); ax.set_ylabel('seasonal index (1 = average week)')
ax.legend(loc='lower left', bbox_to_anchor=(0, 1.02), ncols=3)
ax.set_title(f'Seasonal profile by family (full years {list(map(int, full_years))})', loc='left', pad=26)
viz.source_note(fig, 'Source: Corporacion Favorita (Kaggle).')
fig.tight_layout(rect=(0, 0.03, 1, 1))
fig.savefig(root / 'results/figures/demand_seasonality.png', dpi=150)
plt.show()
si.round(3).describe().loc[['min', 'max']]

C:\Users\Gideon\AppData\Local\Temp\ipykernel_18332\4245794226.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


family,DAIRY,DELI,PREPARED FOODS
min,0.905,0.895,0.949
max,1.095,1.108,1.193


## Variability, intermittency and trend (table view)

In [4]:
def lag_corr(x, k=52):
    x = np.asarray(x, float)
    return float(np.corrcoef(x[k:], x[:-k])[0, 1])
rows = []
for sku, g in panel.groupby('sku_id'):
    u = g.units.to_numpy()
    first, last = u[:52].mean(), u[-52:].mean()
    rows.append({'sku_id': sku, 'name': plant[sku]['name'], 'family': g.family.iloc[0], 'item_nbr': g.item_nbr.iloc[0],
                 'mean_units_wk': round(u.mean()), 'cv': round(u.std() / u.mean(), 3),
                 'zero_week_share': round((u == 0).mean(), 3), 'trend_last_vs_first_year': round(last / first, 2),
                 'lag52_autocorr': round(lag_corr(u), 2), 'weeks_with_closure_day': int((g.n_days < 7).sum()), 'gap_weeks_imputed': int(g.gap_imputed.sum())})
stats = pd.DataFrame(rows)
stats.to_csv(root / 'results/forecast/demand_profile.csv', index=False)
stats

,sku_id,name,family,item_nbr,mean_units_wk,cv,zero_week_share,trend_last_vs_first_year,lag52_autocorr,weeks_with_closure_day,gap_weeks_imputed
0,SKU-01,Plain yogurt cup,DAIRY,939207,5192,0.171,0.000,0.95,0.32,4,6
1,SKU-02,Strawberry yogurt cup,DAIRY,1083196,4716,0.123,0.000,0.85,0.34,4,0
2,SKU-03,Cream cheese spread,DAIRY,838216,4652,0.148,0.000,0.72,0.51,4,0
3,SKU-04,Dessert pudding cup,DAIRY,1083152,4523,0.121,0.000,0.87,0.37,4,0
4,SKU-05,Classic hummus,DELI,866927,5303,0.196,0.000,1.45,0.43,4,3
5,SKU-06,Tzatziki dip,DELI,828630,4742,0.152,0.000,1.10,0.19,4,3
6,SKU-07,Basil pesto,DELI,1121615,2877,0.252,0.027,1.39,0.27,4,0
7,SKU-08,Guacamole,DELI,319093,2692,0.223,0.000,1.00,0.19,4,0
8,SKU-09,Pasta salad,PREPARED FOODS,885543,5423,0.266,0.000,1.91,0.81,4,0
9,SKU-10,Egg salad,PREPARED FOODS,514327,4994,0.102,0.000,0.98,0.13,4,6


## Notes for planning (filled from the table above; see next cell)
These observations feed T3 (forecasting) and T2.2.2 (capacity calibration).

In [5]:
print('CV range:', stats.cv.min(), '-', stats.cv.max())
print('max zero-week share:', stats.zero_week_share.max())
print('trend (last/first year) range:', stats.trend_last_vs_first_year.min(), '-', stats.trend_last_vs_first_year.max())
print('lag-52 autocorrelation range:', stats.lag52_autocorr.min(), '-', stats.lag52_autocorr.max())
print('weeks containing a store-closure day:', stats.weeks_with_closure_day.max())

CV range: 0.083 - 0.359
max zero-week share: 0.027
trend (last/first year) range: 0.72 - 1.91
lag-52 autocorrelation range: 0.13 - 0.81
weeks containing a store-closure day: 4
